# Challenge Telecom X - Parte 2: Análise de Evasão de Clientes (Churn)

**Objetivo:** Construir modelos preditivos para prever a evasão de clientes em uma empresa de telecomunicações.

**Estrutura do Projeto:**
1. Preparação e Exploração dos Dados
2. Análise de Correlação e Seleção de Variáveis
3. Criação e Avaliação de Modelos Preditivos
4. Interpretação e Conclusões

---

## 1. Importação de Bibliotecas e Configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score, accuracy_score, recall_score, precision_score
)
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Bibliotecas importadas com sucesso!")

## 2. Carregamento e Exploração dos Dados

In [ ]:
# Carregar o dataset tratado
df = pd.read_csv('/home/ubuntu/telecomx_churn_clean.csv')

print(f"Shape do dataset: {df.shape}")
print(f"\nPrimeiras linhas:")
print(df.head())
print(f"\nTipos de dados:")
print(df.dtypes)
print(f"\nInformações gerais:")
print(df.info())

In [ ]:
# Análise da variável alvo (Churn)
print("\n=== ANÁLISE DA VARIÁVEL ALVO (CHURN) ===")
print(f"\nDistribuição de Churn:")
print(df['Churn'].value_counts())
print(f"\nProporção (%) de Churn:")
print(df['Churn'].value_counts(normalize=True) * 100)

# Visualizar distribuição
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de contagem
df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição de Churn (Contagem)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Quantidade')
axes[0].set_xticklabels(['Não Evadiu (0)', 'Evadiu (1)'], rotation=0)

# Gráfico de pizza
df['Churn'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                  colors=['#2ecc71', '#e74c3c'], labels=['Não Evadiu', 'Evadiu'])
axes[1].set_title('Proporção de Churn (%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\n✓ Taxa de evasão: {(df['Churn'].sum() / len(df) * 100):.2f}%")

In [ ]:
# Verificar valores faltantes
print("\n=== VERIFICAÇÃO DE VALORES FALTANTES ===")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("✓ Nenhum valor faltante encontrado!")

# Estatísticas descritivas
print("\n=== ESTATÍSTICAS DESCRITIVAS ===")
print(df.describe())

## 3. Preparação dos Dados

In [ ]:
# Preparar dados para modelagem
# Remover coluna de ID de cliente
df_model = df.drop(['customerID'], axis=1)

print(f"Shape após remover ID: {df_model.shape}")
print(f"\nColunas: {df_model.columns.tolist()}")

# Separar variáveis independentes e dependentes
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nVariáveis numéricas: {X.select_dtypes(include=[np.number]).columns.tolist()}")
print(f"Variáveis categóricas: {X.select_dtypes(include=['object']).columns.tolist()}")

In [ ]:
# Codificar variáveis categóricas
X_encoded = X.copy()
label_encoders = {}

categorical_cols = X.select_dtypes(include=['object']).columns

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    print(f"✓ {col}: {len(le.classes_)} classes")

print(f"\n✓ Todas as variáveis codificadas!")
print(f"\nX_encoded shape: {X_encoded.shape}")
print(X_encoded.head())

## 4. Análise de Correlação e Seleção de Variáveis

In [ ]:
# Calcular correlação com a variável alvo
correlation_with_target = X_encoded.corrwith(y).sort_values(ascending=False)

print("\n=== CORRELAÇÃO COM CHURN ===")
print(correlation_with_target)

# Visualizar correlação
plt.figure(figsize=(10, 8))
correlation_with_target.plot(kind='barh', color=['#e74c3c' if x < 0 else '#2ecc71' for x in correlation_with_target.values])
plt.title('Correlação das Variáveis com Churn', fontsize=14, fontweight='bold')
plt.xlabel('Correlação de Pearson')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação completa
plt.figure(figsize=(16, 12))
correlation_matrix = X_encoded.corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação entre Variáveis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Matriz de correlação visualizada!")

In [ ]:
# Selecionar variáveis com maior correlação com Churn (threshold > 0.1 em valor absoluto)
threshold = 0.1
important_features = correlation_with_target[abs(correlation_with_target) > threshold].index.tolist()

print(f"\n=== SELEÇÃO DE VARIÁVEIS ===")
print(f"Threshold de correlação: {threshold}")
print(f"Variáveis selecionadas ({len(important_features)}):")
for feat in important_features:
    print(f"  - {feat}: {correlation_with_target[feat]:.4f}")

# Usar todas as variáveis para modelagem (não fazer seleção muito restritiva)
X_model = X_encoded.copy()
print(f"\n✓ Usando {X_model.shape[1]} variáveis para modelagem")

## 5. Divisão dos Dados

In [ ]:
# Dividir dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y, test_size=0.2, random_state=42, stratify=y
)

print("\n=== DIVISÃO DOS DADOS ===")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print(f"\nProporção de Churn no treino: {y_train.mean():.4f}")
print(f"Proporção de Churn no teste: {y_test.mean():.4f}")

# Normalizar dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Dados normalizados!")

## 6. Criação e Treinamento de Modelos

In [ ]:
# Dicionário para armazenar modelos e resultados
models = {}
results = {}

print("\n=== TREINAMENTO DE MODELOS ===")

# 1. Regressão Logística
print("\n1. Regressão Logística...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
models['Regressão Logística'] = lr_model
print("✓ Treinado!")

# 2. Árvore de Decisão
print("\n2. Árvore de Decisão...")
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_model.fit(X_train, y_train)
models['Árvore de Decisão'] = dt_model
print("✓ Treinado!")

# 3. Random Forest
print("\n3. Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
rf_model.fit(X_train, y_train)
models['Random Forest'] = rf_model
print("✓ Treinado!")

print("\n✓ Todos os modelos treinados com sucesso!")

## 7. Avaliação dos Modelos

In [ ]:
# Função para avaliar modelos
def evaluate_model(model, X_train_eval, X_test_eval, y_train, y_test, model_name, use_scaled=False):
    """
    Avalia um modelo e retorna métricas de desempenho
    """
    # Predições
    y_train_pred = model.predict(X_train_eval)
    y_test_pred = model.predict(X_test_eval)
    
    # Probabilidades (para ROC-AUC)
    y_train_proba = model.predict_proba(X_train_eval)[:, 1]
    y_test_proba = model.predict_proba(X_test_eval)[:, 1]
    
    # Métricas
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    
    return {
        'model_name': model_name,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'train_auc': train_auc,
        'test_auc': test_auc,
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1,
        'y_test_pred': y_test_pred,
        'y_test_proba': y_test_proba
    }

# Avaliar Regressão Logística
results['Regressão Logística'] = evaluate_model(
    models['Regressão Logística'], X_train_scaled, X_test_scaled, y_train, y_test, 
    'Regressão Logística', use_scaled=True
)

# Avaliar Árvore de Decisão
results['Árvore de Decisão'] = evaluate_model(
    models['Árvore de Decisão'], X_train, X_test, y_train, y_test, 'Árvore de Decisão'
)

# Avaliar Random Forest
results['Random Forest'] = evaluate_model(
    models['Random Forest'], X_train, X_test, y_train, y_test, 'Random Forest'
)

print("✓ Modelos avaliados!")

In [ ]:
# Comparar resultados dos modelos
print("\n=== COMPARAÇÃO DE MODELOS ===")
print("\n{:<20} {:<12} {:<12} {:<10} {:<10} {:<10} {:<10}".format(
    'Modelo', 'Train Acc', 'Test Acc', 'Test AUC', 'Precision', 'Recall', 'F1-Score'
))
print("-" * 90)

for model_name, result in results.items():
    print("{:<20} {:<12.4f} {:<12.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
        model_name,
        result['train_accuracy'],
        result['test_accuracy'],
        result['test_auc'],
        result['precision'],
        result['recall'],
        result['f1']
    ))

In [ ]:
# Visualizar comparação de métricas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Comparação de Métricas entre Modelos', fontsize=16, fontweight='bold')

metrics = ['test_accuracy', 'test_auc', 'precision', 'recall', 'f1', 'train_accuracy']
metric_labels = ['Acurácia Teste', 'AUC Teste', 'Precisão', 'Recall', 'F1-Score', 'Acurácia Treino']

for idx, (metric, label) in enumerate(zip(metrics, metric_labels)):
    ax = axes[idx // 3, idx % 3]
    values = [results[model][metric] for model in results.keys()]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    ax.bar(results.keys(), values, color=colors)
    ax.set_title(label, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.set_ylabel('Score')
    for i, v in enumerate(values):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Análise Detalhada dos Modelos

In [ ]:
# Matrizes de Confusão
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Matrizes de Confusão', fontsize=14, fontweight='bold')

for idx, (model_name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['y_test_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(model_name, fontweight='bold')
    axes[idx].set_ylabel('Verdadeiro')
    axes[idx].set_xlabel('Predito')
    axes[idx].set_xticklabels(['Não Evadiu', 'Evadiu'])
    axes[idx].set_yticklabels(['Não Evadiu', 'Evadiu'])

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Curvas ROC', fontsize=14, fontweight='bold')

for idx, (model_name, result) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, result['y_test_proba'])
    auc = result['test_auc']
    
    axes[idx].plot(fpr, tpr, color='#3498db', lw=2, label=f'ROC (AUC = {auc:.3f})')
    axes[idx].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Aleatório')
    axes[idx].set_xlabel('Taxa de Falsos Positivos')
    axes[idx].set_ylabel('Taxa de Verdadeiros Positivos')
    axes[idx].set_title(model_name, fontweight='bold')
    axes[idx].legend(loc='lower right')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Relatórios de Classificação
print("\n=== RELATÓRIOS DE CLASSIFICAÇÃO ===")

for model_name, result in results.items():
    print(f"\n{'='*50}")
    print(f"Modelo: {model_name}")
    print(f"{'='*50}")
    print(classification_report(y_test, result['y_test_pred'], 
                                target_names=['Não Evadiu', 'Evadiu']))

## 9. Importância das Variáveis

In [ ]:
# Importância das variáveis - Árvore de Decisão
dt_importance = pd.DataFrame({
    'feature': X_model.columns,
    'importance': models['Árvore de Decisão'].feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== IMPORTÂNCIA DAS VARIÁVEIS - ÁRVORE DE DECISÃO ===")
print(dt_importance.head(10))

# Visualizar
plt.figure(figsize=(10, 6))
plt.barh(dt_importance.head(10)['feature'], dt_importance.head(10)['importance'], color='#3498db')
plt.xlabel('Importância')
plt.title('Top 10 Variáveis Mais Importantes - Árvore de Decisão', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Importância das variáveis - Random Forest
rf_importance = pd.DataFrame({
    'feature': X_model.columns,
    'importance': models['Random Forest'].feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== IMPORTÂNCIA DAS VARIÁVEIS - RANDOM FOREST ===")
print(rf_importance.head(10))

# Visualizar
plt.figure(figsize=(10, 6))
plt.barh(rf_importance.head(10)['feature'], rf_importance.head(10)['importance'], color='#2ecc71')
plt.xlabel('Importância')
plt.title('Top 10 Variáveis Mais Importantes - Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Coeficientes - Regressão Logística
lr_coef = pd.DataFrame({
    'feature': X_model.columns,
    'coefficient': models['Regressão Logística'].coef_[0]
}).sort_values('coefficient', ascending=False)

print("\n=== COEFICIENTES - REGRESSÃO LOGÍSTICA ===")
print(lr_coef.head(10))

# Visualizar
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in lr_coef.head(10)['coefficient']]
ax.barh(lr_coef.head(10)['feature'], lr_coef.head(10)['coefficient'], color=colors)
ax.set_xlabel('Coeficiente')
ax.set_title('Top 10 Coeficientes - Regressão Logística', fontweight='bold')
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 10. Análise de Fatores que Influenciam a Evasão

In [ ]:
# Análise de variáveis categóricas vs Churn
print("\n=== ANÁLISE DE VARIÁVEIS CATEGÓRICAS VS CHURN ===")

categorical_features = ['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 
                        'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                        'Contract', 'PaperlessBilling', 'PaymentMethod']

# Filtrar apenas as que existem no dataset
categorical_features = [f for f in categorical_features if f in df.columns]

# Calcular taxa de churn por categoria
churn_by_category = {}
for feature in categorical_features:
    churn_rate = df.groupby(feature)['Churn'].agg(['sum', 'count'])
    churn_rate['churn_rate'] = (churn_rate['sum'] / churn_rate['count'] * 100).round(2)
    churn_by_category[feature] = churn_rate
    print(f"\n{feature}:")
    print(churn_rate[['churn_rate']].sort_values('churn_rate', ascending=False))

In [ ]:
# Visualizar taxa de churn por contrato (variável importante)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Contrato
contract_churn = df.groupby('Contract')['Churn'].agg(['sum', 'count'])
contract_churn['rate'] = (contract_churn['sum'] / contract_churn['count'] * 100)
axes[0, 0].bar(contract_churn.index, contract_churn['rate'], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[0, 0].set_title('Taxa de Churn por Tipo de Contrato', fontweight='bold')
axes[0, 0].set_ylabel('Taxa de Churn (%)')
for i, v in enumerate(contract_churn['rate']):
    axes[0, 0].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# Internet Service
internet_churn = df.groupby('InternetService')['Churn'].agg(['sum', 'count'])
internet_churn['rate'] = (internet_churn['sum'] / internet_churn['count'] * 100)
axes[0, 1].bar(internet_churn.index, internet_churn['rate'], color=['#3498db', '#e74c3c', '#95a5a6'])
axes[0, 1].set_title('Taxa de Churn por Tipo de Internet', fontweight='bold')
axes[0, 1].set_ylabel('Taxa de Churn (%)')
for i, v in enumerate(internet_churn['rate']):
    axes[0, 1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# Online Security
security_churn = df.groupby('OnlineSecurity')['Churn'].agg(['sum', 'count'])
security_churn['rate'] = (security_churn['sum'] / security_churn['count'] * 100)
axes[1, 0].bar(security_churn.index, security_churn['rate'], color=['#e74c3c', '#2ecc71'])
axes[1, 0].set_title('Taxa de Churn por Online Security', fontweight='bold')
axes[1, 0].set_ylabel('Taxa de Churn (%)')
for i, v in enumerate(security_churn['rate']):
    axes[1, 0].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# Tech Support
support_churn = df.groupby('TechSupport')['Churn'].agg(['sum', 'count'])
support_churn['rate'] = (support_churn['sum'] / support_churn['count'] * 100)
axes[1, 1].bar(support_churn.index, support_churn['rate'], color=['#e74c3c', '#2ecc71'])
axes[1, 1].set_title('Taxa de Churn por Tech Support', fontweight='bold')
axes[1, 1].set_ylabel('Taxa de Churn (%)')
for i, v in enumerate(support_churn['rate']):
    axes[1, 1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Análise de variáveis numéricas
print("\n=== ANÁLISE DE VARIÁVEIS NUMÉRICAS VS CHURN ===")

numeric_features = ['Tenure', 'MonthlyCharge', 'TotalCharge', 'DailyCharge']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, feature in enumerate(numeric_features):
    ax = axes[idx // 2, idx % 2]
    
    # Boxplot
    df.boxplot(column=feature, by='Churn', ax=ax)
    ax.set_title(f'{feature} por Churn', fontweight='bold')
    ax.set_xlabel('Churn')
    ax.set_ylabel(feature)
    ax.set_xticklabels(['Não Evadiu', 'Evadiu'])
    
    # Estatísticas
    print(f"\n{feature}:")
    print(df.groupby('Churn')[feature].describe())

plt.suptitle('')
plt.tight_layout()
plt.show()

## 11. Conclusões e Recomendações

In [ ]:
# Resumo final
print("\n" + "="*80)
print("RELATÓRIO FINAL - CHALLENGE TELECOM X PARTE 2")
print("="*80)

print("\n1. DATASET")
print("-" * 80)
print(f"   • Total de clientes: {len(df):,}")
print(f"   • Taxa de evasão (Churn): {(df['Churn'].sum() / len(df) * 100):.2f}%")
print(f"   • Clientes que evadiram: {df['Churn'].sum():,}")
print(f"   • Clientes que permaneceram: {(len(df) - df['Churn'].sum()):,}")
print(f"   • Total de variáveis: {X_model.shape[1]}")

print("\n2. DESEMPENHO DOS MODELOS")
print("-" * 80)
best_model = max(results.items(), key=lambda x: x[1]['test_auc'])
print(f"   • Melhor modelo (por AUC): {best_model[0]}")
print(f"   • AUC do melhor modelo: {best_model[1]['test_auc']:.4f}")
print(f"   • Acurácia: {best_model[1]['test_accuracy']:.4f}")
print(f"   • Precisão: {best_model[1]['precision']:.4f}")
print(f"   • Recall: {best_model[1]['recall']:.4f}")
print(f"   • F1-Score: {best_model[1]['f1']:.4f}")

print("\n3. FATORES QUE INFLUENCIAM A EVASÃO")
print("-" * 80)
print("   Variáveis mais importantes (Random Forest):")
for idx, row in rf_importance.head(5).iterrows():
    print(f"   • {row['feature']}: {row['importance']:.4f}")

print("\n4. INSIGHTS PRINCIPAIS")
print("-" * 80)
print("   • Clientes com contrato mês-a-mês têm maior taxa de evasão")
print("   • Clientes com Internet Fiber Optic têm maior taxa de evasão")
print("   • Clientes sem Online Security têm maior taxa de evasão")
print("   • Clientes sem Tech Support têm maior taxa de evasão")
print("   • Tenure (tempo de cliente) é inversamente correlacionado com churn")

print("\n5. RECOMENDAÇÕES")
print("-" * 80)
print("   • Oferecer incentivos para contratos de longo prazo")
print("   • Melhorar a qualidade do serviço Fiber Optic")
print("   • Promover serviços de Online Security e Tech Support")
print("   • Implementar programa de retenção para clientes novos (primeiros 12 meses)")
print("   • Usar o modelo Random Forest para priorizar ações de retenção")

print("\n" + "="*80)

In [ ]:
# Salvar o modelo melhor
print("\n✓ Análise completa!")
print(f"\nMelhor modelo para produção: {best_model[0]}")
print(f"Desempenho esperado em novos dados:")
print(f"  - AUC: {best_model[1]['test_auc']:.4f}")
print(f"  - Acurácia: {best_model[1]['test_accuracy']:.4f}")
print(f"  - Recall: {best_model[1]['recall']:.4f} (importante para identificar clientes em risco)")